In [ ]:
FIGURE = 'Fig.S3'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
import json
import os
from datetime import datetime
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import netCDF4 as nc
import numpy as np
import pandas as pd
ASCII_ROOT = asset_path('asset_4d8b1bf7a531')
FINAL_ROOT = str(OUTPUT / 'Fig.S2-S3')
PATHWAY_ORDER = ['SPEI', 'SSMIrz']
PATHWAY_LABELS = {'SPEI': 'SPEI-SIF', 'SSMIrz': 'SSMI$_{\\mathrm{rz}}$-SIF'}
SEASON_ORDER = ['MAM']
STIPPLE_Q = 0.05

def setup_style():
    mpl.rcParams.update({'font.family': 'Times New Roman', 'font.serif': ['Times New Roman', 'DejaVu Serif'], 'mathtext.fontset': 'stix', 'font.size': 7, 'axes.linewidth': 0.45, 'axes.titlesize': 8, 'xtick.labelsize': 5.5, 'ytick.labelsize': 5.5, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600, 'figure.dpi': 600})

def filled(x, fill_value=np.nan):
    if np.ma.isMaskedArray(x):
        return x.filled(fill_value)
    return np.asarray(x)

def read_season_names(ds):
    return [str(x) for x in nc.chartostring(ds.variables['season_name'][:])] if 'season_name' in ds.variables else SEASON_ORDER

def sort_lon(lon, data):
    lon = np.asarray(lon, dtype=np.float64)
    lon2 = (lon + 180.0) % 360.0 - 180.0
    order = np.argsort(lon2)
    return (lon2[order], data[..., order])

def format_longitude_for_compact_panels(x, pos=None):
    if np.isclose(abs(x), 180.0):
        return '180'
    if np.isclose(x, 0.0):
        return '0°'
    if x < 0:
        return f'{abs(int(round(x)))}°W'
    return f'{int(round(x))}°E'

def add_reference_gridlines(ax, col):
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.35, color='gray', alpha=0.45, linestyle='--', xlocs=np.arange(-180, 181, 60), ylocs=np.arange(-60, 91, 30))
    gl.top_labels = False
    gl.bottom_labels = False
    gl.left_labels = col == 0
    gl.right_labels = col == 1
    gl.xformatter = mticker.FuncFormatter(format_longitude_for_compact_panels)
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 5.8, 'family': 'Times New Roman'}
    gl.ylabel_style = {'size': 5.8, 'family': 'Times New Roman'}
    gl.xpadding = 1.5
    gl.ypadding = 1.5

def add_bottom_longitude_labels(ax):
    for x, lab in [(1.0 / 6.0, '120°W'), (2.0 / 6.0, '60°W'), (3.0 / 6.0, '0°'), (4.0 / 6.0, '60°E'), (5.0 / 6.0, '120°E')]:
        ax.text(x, -0.055, lab, transform=ax.transAxes, ha='center', va='top', fontsize=5.8, fontfamily='Times New Roman', color='black', clip_on=False)

def add_shared_180_labels(fig, bottom_left_ax, bottom_right_ax):
    left_pos = bottom_left_ax.get_position()
    right_pos = bottom_right_ax.get_position()
    y = left_pos.y0 + -0.055 * left_pos.height
    for x, lab in [(left_pos.x0, '180'), ((left_pos.x1 + right_pos.x0) / 2.0, '180'), (right_pos.x1, '180')]:
        fig.text(x, y, lab, ha='center', va='top', fontsize=5.8, fontfamily='Times New Roman', color='black')

def area_weighted_stats(lat, slope, sig):
    valid = np.isfinite(slope)
    pos = valid & (slope > 0)
    neg = valid & (slope < 0)
    denom_mask = pos | neg
    if not np.any(denom_mask):
        return {k: np.nan for k in ['positive_ratio', 'negative_ratio', 'significant_positive_ratio', 'significant_negative_ratio']}
    weights = np.tile(np.cos(np.deg2rad(np.asarray(lat, dtype=np.float64)))[:, None], (1, slope.shape[1]))
    total_area = np.sum(weights[denom_mask])
    return {'positive_ratio': float(np.sum(weights[pos]) / total_area), 'negative_ratio': float(np.sum(weights[neg]) / total_area), 'significant_positive_ratio': float(np.sum(weights[pos & sig]) / total_area), 'significant_negative_ratio': float(np.sum(weights[neg & sig]) / total_area)}

def draw_reference_inset(ax, stats_dict):
    axins = inset_axes(ax, width='30%', height='30%', loc='lower left', bbox_to_anchor=(0.01, 0.17, 0.8, 0.7), bbox_transform=ax.transAxes, borderpad=0.0)
    vals = [stats_dict['positive_ratio'], stats_dict['negative_ratio']]
    sig_vals = [stats_dict['significant_positive_ratio'], stats_dict['significant_negative_ratio']]
    axins.bar(range(2), vals, width=0.6, color=['#ff9999', '#99ccff'], linewidth=0, zorder=1)
    axins.bar(range(2), sig_vals, width=0.6, color=['darkred', 'steelblue'], linewidth=0, zorder=2)
    axins.set_xticks(range(2))
    axins.set_xticklabels(['Positive', 'Negative'], fontsize=5.6, rotation=30)
    axins.set_yticks([])
    axins.set_ylim(0, 1.1)
    axins.set_xlim(-0.5, 1.5)
    axins.tick_params(axis='x', length=0, pad=0.5)
    axins.patch.set_alpha(0.52)
    axins.patch.set_facecolor('white')
    for spine in axins.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.45)
        spine.set_edgecolor('0.35')
    for i in range(2):
        axins.text(i, min(vals[i] + 0.11, 1.04), f'{vals[i] * 100:.2f}%', ha='center', va='bottom', fontsize=5.9, color='black', zorder=4)
        axins.text(i, min(sig_vals[i] + 0.035, 0.98), f'{sig_vals[i] * 100:.2f}%', ha='center', va='bottom', fontsize=5.9, color='black', zorder=4)
    handles = [Rectangle((0, 0), 1, 1, fc='darkred', ec='darkred'), Rectangle((0, 0), 1, 1, fc='steelblue', ec='steelblue')]
    leg = axins.legend(handles=handles, labels=['Sig. Positive', 'Sig. Negative'], fontsize=5.7, loc='upper left', bbox_to_anchor=(-0.02, 1.94), frameon=False, handlelength=0.55, handleheight=0.55, handletextpad=0.28, labelspacing=0.18, borderaxespad=0.0)
    for text in leg.get_texts():
        text.set_fontfamily('Times New Roman')

def plot_yearly(kind):
    if kind == 'rmax':
        files = {'SPEI': str(INPUT_DIR / 'SPEI_Rmax.nc'), 'SSMIrz': str(INPUT_DIR / 'SSMIrz_Rmax.nc')}
        out_dir = os.path.join(FINAL_ROOT, '01_rmax_trend_all_valid')
        levels = np.linspace(-0.05, 0.05, 11)
        ticks = np.linspace(-0.05, 0.05, 11)
        prefix = 'FigS_TheilSen_Rmaxtrend_all_valid_pixels_peffstippling_yearly_vlim0p1'
    else:
        files = {'SPEI': str(INPUT_DIR / 'SPEI_Ts.nc'), 'SSMIrz': str(INPUT_DIR / 'SSMIrz_Ts.nc')}
        out_dir = os.path.join(FINAL_ROOT, '02_optscale_trend_jump_filter')
        levels = np.linspace(-2.0, 2.0, 11)
        ticks = np.linspace(-2.0, 2.0, 11)
        prefix = 'FigS_TheilSen_optimal_timescale_trend_Tsjump_filtering_peffstippling_yearly_vlim1'
    os.makedirs(out_dir, exist_ok=True)
    cmap = mpl.cm.RdBu_r
    norm = mpl.colors.BoundaryNorm(levels, cmap.N, extend='both')
    fig = plt.figure(figsize=(7.05, 1.3 * len(SEASON_ORDER) + 1.25))
    gs = fig.add_gridspec(nrows=len(SEASON_ORDER), ncols=2, left=0.045, right=0.955, top=0.91, bottom=0.13, wspace=0.018, hspace=0.082)
    panel_letters = list('abcdefgh')
    mesh = None
    stats_rows = []
    bottom_left_ax = None
    bottom_right_ax = None
    for row, season in enumerate(SEASON_ORDER):
        for col, pathway in enumerate(PATHWAY_ORDER):
            with open_nc(files[pathway]) as ds:
                lat = np.asarray(ds.variables['lat'][:])
                lon = np.asarray(ds.variables['lon'][:])
                seasons = ['MAM', 'JJA', 'SON', 'DJF']
                si = {s: i for i, s in enumerate(seasons)}[season]
                slope = filled(ds.variables['slope'][si], np.nan).astype(np.float32)
                q_fdr = filled(ds.variables['p_eff'][si], np.nan).astype(np.float32)
                mask = filled(ds.variables['analysis_mask'][si], 0).astype(np.uint8) == 1
                slope = np.where(mask, slope, np.nan)
                sig = mask & np.isfinite(slope) & np.isfinite(q_fdr) & (q_fdr < STIPPLE_Q)
                lon_s, slope_s = sort_lon(lon, slope)
                _, sig_s = sort_lon(lon, sig.astype(np.int8))
                ax = fig.add_subplot(gs[row, col], projection=ccrs.PlateCarree())
                ax.set_anchor('E' if col == 0 else 'W')
                ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())
                ax.add_feature(cfeature.LAND, facecolor='#F2F1DF', edgecolor='black', linewidth=0.25, zorder=0)
                ax.coastlines(linewidth=0.25, color='black', zorder=6)
                mesh = ax.contourf(lon_s, lat, slope_s, levels=levels, cmap=cmap, norm=norm, extend='both', transform=ccrs.PlateCarree(), zorder=1)
                step = 2
                yy, xx = np.where(sig_s[::step, ::step] == 1)
                if yy.size:
                    ax.scatter(lon_s[::step][xx], lat[::step][yy], s=0.36, c='white', alpha=0.95, linewidths=0, transform=ccrs.PlateCarree(), rasterized=True, zorder=4)
                    ax.scatter(lon_s[::step][xx], lat[::step][yy], s=0.12, c='black', alpha=0.98, linewidths=0, transform=ccrs.PlateCarree(), rasterized=True, zorder=5)
                add_reference_gridlines(ax, col)
                if row == len(SEASON_ORDER) - 1:
                    add_bottom_longitude_labels(ax)
                    if col == 0:
                        bottom_left_ax = ax
                    else:
                        bottom_right_ax = ax
                for spine in ax.spines.values():
                    spine.set_linewidth(0.45)
                    spine.set_edgecolor('0.35')
                letter = panel_letters[row + col * len(SEASON_ORDER)]
                ax.text(0.018, 0.965, letter, transform=ax.transAxes, ha='left', va='top', fontsize=8.6, fontfamily='Arial', fontweight='normal', color='black', zorder=10)
                ax.text(-176, 45, season, transform=ccrs.PlateCarree(), ha='left', va='center', fontsize=8.4, fontfamily='Times New Roman', fontweight='normal', color='black', zorder=10)
                if row == 0:
                    ax.text(0.5, 1.025, PATHWAY_LABELS[pathway], transform=ax.transAxes, ha='center', va='bottom', fontsize=9.5, fontweight='bold')
                st = area_weighted_stats(lat, slope, sig)
                draw_reference_inset(ax, st)
                stats_rows.append({'kind': kind, 'pathway': pathway, 'season': season, 'positive_all_area_percent': st['positive_ratio'] * 100, 'negative_all_area_percent': st['negative_ratio'] * 100, 'significant_positive_peff_area_percent': st['significant_positive_ratio'] * 100, 'significant_negative_peff_area_percent': st['significant_negative_ratio'] * 100})
    fig.canvas.draw()
    if bottom_left_ax is not None and bottom_right_ax is not None:
        add_shared_180_labels(fig, bottom_left_ax, bottom_right_ax)
    cax = fig.add_axes([0.245, 0.065, 0.51, 0.024])
    cb = fig.colorbar(mesh, cax=cax, orientation='horizontal', extend='both', ticks=ticks)
    cb.outline.set_linewidth(0.45)
    cb.set_label('Rmax trend (year$^{-1}$)' if kind == 'rmax' else 'Ts trend (months year$^{-1}$)', fontsize=7, labelpad=1)
    fig.text(0.5, 0.01, 'Dots: $p_{\\mathrm{eff}}<0.05$', ha='center', fontsize=7)
    cb.ax.tick_params(length=2.2, width=0.45, labelsize=7.0, pad=1.0)
    png = os.path.join(out_dir, prefix + '_600dpi.png')
    tif = os.path.join(out_dir, prefix + '_600dpi.tif')
    pdf = os.path.join(out_dir, prefix + '.pdf')
    csv = os.path.join(out_dir, prefix + '_panel_statistics.csv')
    fig.savefig(png, dpi=600)
    fig.savefig(tif, dpi=600)
    fig.savefig(pdf)
    plt.close(fig)
    pd.DataFrame(stats_rows).to_csv(csv, index=False, encoding='utf-8-sig')
    return {'png': png, 'tif': tif, 'pdf': pdf, 'csv': csv, 'levels': levels.tolist(), 'ticks': ticks.tolist()}

def main():
    setup_style()
    plot_yearly('optscale')
if __name__ == '__main__':
    main()
